# Watsonx Demand Forecasting BI - Demo

End-to-end walkthrough of the demand forecasting pipeline:
synthetic data generation, feature engineering, statistical models,
and ensemble forecasting with Brazilian calendar awareness.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.feature_engineering import add_lag_features, add_rolling_features, add_calendar_features
from src.data.calendar_br import get_national_holidays, get_commercial_dates
from src.models.statistical import ARIMAForecaster, ETSForecaster
from src.models.ensemble import EnsembleForecaster

pd.set_option("display.max_columns", 20)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

## 1. Synthetic Retail Data

Generate 2 years of daily sales data with:
- Linear upward trend
- Weekly seasonality (lower on weekends)
- Annual seasonality
- Random noise

In [ ]:
np.random.seed(42)
days = 730
t = np.arange(days)

trend = 200 + 0.15 * t
weekly = 30 * np.sin(2 * np.pi * t / 7)
annual = 50 * np.sin(2 * np.pi * t / 365)
noise = np.random.normal(0, 15, days)

sales = np.maximum(0, trend + weekly + annual + noise)

dates = pd.date_range("2023-01-01", periods=days, freq="D")
df = pd.DataFrame({"date": dates, "sku": "PROD-001", "quantity": sales})

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["date"], df["quantity"], linewidth=0.8)
ax.set_title("Synthetic Daily Sales - PROD-001")
ax.set_xlabel("Date")
ax.set_ylabel("Units Sold")
plt.tight_layout()
plt.show()

print(f"Shape: {df.shape}")
print(f"Date range: {df.date.min()} to {df.date.max()}")
df.head()

## 2. Feature Engineering

Add temporal features: lag values, rolling statistics, and calendar indicators.

In [ ]:
# Lag features
df_feat = add_lag_features(df, target_col="quantity", lags=[1, 7, 14, 28], group_col="sku")
print("Lag features:")
print([c for c in df_feat.columns if "lag" in c])

# Rolling features
df_feat = add_rolling_features(df_feat, target_col="quantity", windows=[7, 14, 30], statistics=["mean", "std"], group_col="sku")
print("
Rolling features:")
print([c for c in df_feat.columns if "roll" in c])

# Calendar features
df_feat = add_calendar_features(df_feat)
print("
Calendar features:")
print([c for c in df_feat.columns if c in ["day_of_week", "month", "is_weekend", "quarter"]])

df_feat.tail()

## 3. Brazilian Calendar

Detect national holidays and commercial dates for 2025-2026.

In [ ]:
for year in [2025, 2026]:
    holidays = get_national_holidays(year)
    commercial = get_commercial_dates(year)
    print(f"
=== {year} ===")
    print(f"National holidays: {len(holidays)}")
    for d, name in sorted(holidays.items()):
        print(f"  {d}: {name}")
    print(f"
Commercial dates: {len(commercial)}")
    key_dates = {d: n for d, n in commercial.items() if n in ["Dia das Maes", "Dia dos Namorados", "Dia dos Pais", "Black Friday BR"]}
    for d, name in sorted(key_dates.items()):
        print(f"  {d}: {name}")

## 4. Statistical Models

Fit ARIMA and ETS models on the synthetic data and generate 30-day forecasts.

In [ ]:
# Prepare series (use last 365 days for fitting)
train = df["quantity"].iloc[-365:]
train.index = pd.date_range("2024-01-01", periods=365, freq="D")
horizon = 30

# ARIMA
arima = ARIMAForecaster(order=(1, 1, 1), auto_order=False, seasonal_order=(0, 0, 0, 0))
arima.fit(train)
arima_fc = arima.predict(horizon=horizon)

# ETS
ets = ETSForecaster(seasonal_periods=7)
ets.fit(train)
ets_fc = ets.predict(horizon=horizon)

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-60:], train.values[-60:], label="Historical", color="black")
fc_dates = pd.date_range(train.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")
ax.plot(fc_dates, arima_fc.values, label="ARIMA", linestyle="--")
ax.plot(fc_dates, ets_fc.values, label="ETS", linestyle="--")
ax.legend()
ax.set_title("30-Day Forecast Comparison")
ax.set_xlabel("Date")
ax.set_ylabel("Units")
plt.tight_layout()
plt.show()

print(f"ARIMA forecast mean: {arima_fc.mean():.1f}")
print(f"ETS forecast mean: {ets_fc.mean():.1f}")

## 5. Ensemble Forecasting

Combine ARIMA and ETS predictions using inverse-error weighted averaging.

In [ ]:
# Build ensemble
ensemble = EnsembleForecaster(models=[arima, ets], method="weighted_average")

# Set backtest scores (simulated RMSE values)
ensemble.set_backtest_scores({"ARIMA": 18.5, "ETS": 15.2})

print("Ensemble weights:", ensemble.weights)
print("Ensemble summary:", ensemble.summary())

# Get individual + ensemble forecasts
results = ensemble.get_individual_forecasts(horizon=horizon)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-60:], train.values[-60:], label="Historical", color="black")
for col in results.columns:
    style = "-" if col == "ensemble" else "--"
    lw = 2 if col == "ensemble" else 1
    ax.plot(fc_dates, results[col].values, label=col, linestyle=style, linewidth=lw)
ax.legend()
ax.set_title("Ensemble Forecast vs Individual Models")
plt.tight_layout()
plt.show()

## 6. Architecture

The pipeline follows a modular design:

1. **Data Sources**: POS/ERP data, external signals, Brazilian calendar
2. **Feature Engineering**: Lag features, rolling statistics, calendar encoding
3. **Models**: ARIMA, ETS, XGBoost/LightGBM, Granite-TSFM (IBM Watsonx)
4. **Ensemble**: Inverse-error weighted combination with backtesting
5. **Output**: Point forecasts, confidence intervals, dashboard integration

See  for the full Mermaid diagram and detailed documentation.